In [ ]:
# 1. Установка + клон
!pip install -q torch numpy faiss-cpu tokenizers datasets scipy loguru
!git clone https://github.com/BlackCatSpb/FCF.git /content/FCF 2>/dev/null
%cd /content/FCF
!git pull 2>/dev/null
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/FCF/checkpoints
print('Ready')

In [ ]:
# 2. Загрузка модели (новая или из чекпоинта)
import sys, os, torch
sys.path.insert(0, '.')
from fcf.config import load_config
from fcf.primordial_layer import PrimordialLayer
from fcf.language_trainer import LanguageTrainer
from fcf.tokenizer_utils import load_tokenizer
from fcf.utils import load_primordial_layer, save_primordial_layer

DRIVE = '/content/drive/MyDrive/FCF'
CKPT = f'{DRIVE}/checkpoints/wiki_latest'

tokenizer = load_tokenizer('tokenizer.json')
print(f'Tokenizer: {tokenizer.get_vocab_size()} words')

if os.path.exists(f'{CKPT}/weights.pt'):
    layer = load_primordial_layer(CKPT, PrimordialLayer)
    print(f'Resumed from Drive: step={layer.meta.usage_count}')
else:
    layer = PrimordialLayer(load_config())
    print(f'New layer: {sum(p.numel() for p in layer.parameters()):,} params')

trainer = LanguageTrainer(
    layer=layer, tokenizer=tokenizer,
    checkpoint_dir=f'{DRIVE}/checkpoints/wiki_latest'
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# 3. Обучение (30K шагов, автосохранение на Drive каждые 1000)
STEPS = 30000
print(f'Training {STEPS} steps on {device}...')
stats = trainer.train(max_steps=STEPS, device=device, use_wikipedia=True)
print(f'Done. SRG: {stats.get("average_confidence", 0):.3f}')

save_primordial_layer(layer, CKPT)
print(f'Saved to Drive: {CKPT}')

In [ ]:
# 4. Статистика
print(f'Steps total: {layer.meta.usage_count}')
print(f'Confidence: {layer.meta.average_confidence():.3f}')
print(f'Snapshots: {len(layer.state_storage)}')
print(f'Drive checkpoints:')
!ls -lh /content/drive/MyDrive/FCF/checkpoints/wiki_latest/ 2>/dev/null